<a href="https://colab.research.google.com/github/manuellazarte/Procesamiento-del-Lenguaje-Natural/blob/main/src/Trabajo_pr%C3%A1ctico_N2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [57]:
!pip install pgvector
!pip install nltk
!pip install gensim
!pip install requests
!pip install sentence-transformers


[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


     ---------------------------------------- 0.0/740.6 kB ? eta -:--:--
     ------------------------------------  737.3/740.6 kB 15.5 MB/s eta 0:00:01
     ------------------------------------- 740.6/740.6 kB 15.5 MB/s eta 0:00:00
  Using cached huggingface_hub-1.33.0-py3-none-any.whl (846 kB)
     ---------------------------------------- 0.0/124.1 MB ? eta -:--:--
      -------------------------------------- 1.6/124.1 MB 52.2 MB/s eta 0:00:03
     - ------------------------------------- 3.4/124.1 MB 43.7 MB/s eta 0:00:03
     - ------------------------------------- 5.2/124.1 MB 41.4 MB/s eta 0:00:03
     -- ------------------------------------ 7.0/124.1 MB 40.4 MB/s eta 0:00:03
     -- ------------------------------------ 8.8/124.1 MB 40.1 MB/s eta 0:00:03
     --- ---------------------------------- 10.6/124.1 MB 38.6 MB/s eta 0:00:03
     --- ---------------------------------- 12.3/124.1 MB 38.6 MB/s eta 0:00:03
     ---- --------------------------------- 14.1/124.1 MB 38.5 MB/s et


[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## A. Recuperar y preparar el corpus

In [59]:
import re
import os
import nltk
import requests
import unicodedata
import numpy as np
import pandas as pd
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from gensim.models import Word2Vec, KeyedVectors
from sentence_transformers import SentenceTransformer, util

In [ ]:
# Ruta para Local y Colab, descomentar la que haga falta
# df = pd.read_csv("/content/libros.csv")
df = pd.read_csv("../data/libros.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   book_id           150 non-null    int64 
 1   titulo            150 non-null    object
 2   autores           150 non-null    object
 3   generos           150 non-null    object
 4   serie             48 non-null     object
 5   sinopsis          150 non-null    object
 6   url_libro         150 non-null    object
 7   categoria_origen  150 non-null    object
 8   fecha_extraccion  150 non-null    object
 9   url_portada       150 non-null    object
dtypes: int64(1), object(9)
memory usage: 11.8+ KB


In [16]:
df.head()

,book_id,titulo,autores,generos,serie,sinopsis,url_libro,categoria_origen,fecha_extraccion,url_portada
0,125386,Batallón disciplinario,['Karl von Vereiter'],"['Aventuras', 'Bélico', 'Novela']",NaN,De todas las unidades especiales que la Wehrma...,/book/batallon-disciplinario/,belico,2026-09-07,https://assets.lectulandia.com/b/ab/Karl%20von...
1,125355,Bombardero,['Len Deighton'],"['Bélico', 'Intriga', 'Novela']",NaN,"En una calurosa noche de junio de 1943, una fu...",/book/bombardero/,belico,2026-09-07,https://assets.lectulandia.com/b/ab/Len%20Deig...
2,125294,Las quimeras negras,['Jean Lartéguy'],"['Bélico', 'Novela']",NaN,En medio de un África presa de todos los delir...,/book/las-quimeras-negras/,belico,2026-09-07,https://assets.lectulandia.com/b/ab/Jean%20Lar...
3,125082,Tierra de asilo,['Jean Lartéguy'],"['Bélico', 'Novela']",NaN,Sauveterre es un rincón de la Francia olvidada...,/book/tierra-de-asilo/,belico,2026-09-07,https://assets.lectulandia.com/b/ab/Jean%20Lar...
4,124552,Colina 112,['Adrian Goldsworthy'],"['Bélico', 'Histórico', 'Novela']",NaN,"Se avecina el día D. Sin embargo, para tres jó...",/book/colina-112/,belico,2026-09-07,https://assets.lectulandia.com/b/ab/Adrian%20G...


### Preprocesamiento

#### Configuracion para la limpieza

In [34]:
# ///////////// Configuracion NLTK /////////////
# Descargar el conjunto de palabras de parada en español
nltk.download('stopwords', quiet=True)  #`quiet=True` para silenciar la salida de la celda
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)  # Esto hace falta porque NLTK se actualizo, si no se hace esto,
                                        # da error de que no encuentra el tokenizador de palabras

# Definir las palabras de parada en español
stop_words = set(stopwords.words('spanish'))

# Función para eliminar las palabras de parada de una frase
def remove_stopwords(text):
    word_tokens = word_tokenize(text)
    filtered_text = [word for word in word_tokens if word.casefold() not in stop_words]
    return " ".join(filtered_text)

# ///////////// Configuracion NLTK /////////////

# Definimos funcion para remover acentos
def remove_accents(input_str):
    nfkd_form = unicodedata.normalize('NFKD', input_str)
    return ''.join([c for c in nfkd_form if not unicodedata.combining(c)])

# Definimos el preprocesador para aplicar la limpieza
def preprocesar_lexico(texto):
    texto = texto.lower()                   # minisculas
    texto = re.sub(r"[^\w\s]", " ", texto)  # sin puntuación
    texto = remove_stopwords(texto)         # sin stopwords
    texto = remove_accents(texto)           # sin acentos
    return " ".join(texto.split())

Limpieza del texto en crudo aplicada en una nueva columna limpia

In [35]:
df['sinopsis_limpia'] = df['sinopsis'].apply(preprocesar_lexico)

print("Sinopsis en crudo:")
print(df['sinopsis'])

print("\nSinopsis limpia:")
print(df['sinopsis_limpia'])

Sinopsis en crudo:
0      De todas las unidades especiales que la Wehrma...
1      En una calurosa noche de junio de 1943, una fu...
2      En medio de un África presa de todos los delir...
3      Sauveterre es un rincón de la Francia olvidada...
4      Se avecina el día D. Sin embargo, para tres jó...
                             ...                        
145    Alfa Ndiaye es senegalés y ha acabado combatie...
146    Veinte años después de la derrota del ejército...
147    Un día, una niña de apenas cinco años preguntó...
148    En esta sensacional novela sobre pilotos naval...
149    En la madrugada del 30 de abril de 1945, el ra...
Name: sinopsis, Length: 150, dtype: object

Sinopsis limpia:
0      todas unidades especiales wehrmacht poseyo tra...
1      calurosa noche junio 1943 fuerza 700 bombarder...
2      medio africa presa delirios tres hombres consu...
3      sauveterre rincon francia olvidada cerril auto...
4      avecina dia d embargo tres jovenes gales sur g...
        

### Tokenizacion

In [36]:
# Creamos la lista de oraciones tokenizadas
corpus_tokens = [texto.split() for texto in df['sinopsis_limpia']]

## B. Word2Vec / FastText: propio contra pre-entrenado

In [ ]:
# posible modelo
modelo_propio = Word2Vec(
    sentences = corpus_tokens,
    vector_size = 300,    # es igual al modelo referencia de la catedra
    window = 5,           # ventana moderadamente amplia porque al ser sinopsis las relaciones no son consecutivas
    min_count = 3,        # poner igual a 1 haria que no aprenda toda su distrubucion, poner igual 5 haria que descate mucho vocabulario, se busca punto medio
    sg = 1,               # skip-gram para no ignrar palabras poco frecuentes que puedan ser importantes
    epochs = 30,          # como es un copus pequeño necesito muchas pasadas para converger
    seed = 42,            # inicializacion
    workers = 1           # un solo hilo para fijar la seed
)

print(f"Palabras en el vocabulario propio: {len(modelo_propio.wv)}")

Palabras en el vocabulario propio: 1185


In [51]:
sbw_file = "SBW-vectors-300-min5.bin.gz"
url = 'https://cs.famaf.unc.edu.ar/~ccardellino/SBWCE/SBW-vectors-300-min5.bin.gz'


# Descargamos el modelo pre-entrenado si no existe en Colab
if not os.path.exists(sbw_file):
    print("Descargando modelo SBW-vectors-300-min5 (aprox. 500 MB)...")
    with requests.get(url, stream=True) as response: # stream=True para descargar en bloques
        response.raise_for_status()  # Lanza excepción si el link falla
        with open(sbw_file, "wb") as f:
            for chunk in response.iter_content(chunk_size=1024 * 1024):  # 1 MB por bloque
                if chunk:
                    f.write(chunk)

# Carga con Gensim KeyedVectors
modelo_sbw = KeyedVectors.load_word2vec_format(sbw_file, binary=True)
print(f"Modelo SBW cargado. Vocabulario: {len(modelo_sbw.key_to_index)} palabras.")

Modelo SBW cargado. Vocabulario: 1000653 palabras.


In [52]:
# Palabras clave del dominio para comparar
palabras_dominio = ['guerra', 'soldado', 'ejercito', 'batalla']

comparacion = []

for palabra in palabras_dominio:
    # Vecinos en el modelo propio
    if palabra in modelo_propio.wv:
        vecinos_propio = [f"{w} ({sim:.2f})" for w, sim in modelo_propio.wv.most_similar(palabra, topn=4)]
    else:
        vecinos_propio = ["OOV (fuera de vocabulario)"]

    # Vecinos en el modelo pre-entrenado (SBW)
    if palabra in modelo_sbw:
        vecinos_sbw = [f"{w} ({sim:.2f})" for w, sim in modelo_sbw.most_similar(palabra, topn=4)]
    else:
        vecinos_sbw = ["OOV"]

    comparacion.append({
        "Palabra": palabra,
        "Vecinos Modelo Propio": ", ".join(vecinos_propio),
        "Vecinos Modelo SBW (Pre-entrenado)": ", ".join(vecinos_sbw)
    })

df_vecinos = pd.DataFrame(comparacion)
df_vecinos

,Palabra,Vecinos Modelo Propio,Vecinos Modelo SBW (Pre-entrenado)
0,guerra,"segunda (0.95), mundial (0.95), bolchevique (0...","Guerra (0.74), guerras (0.69), francoprusiana ..."
1,soldado,"raso (0.94), regimiento (0.91), semanas (0.91)...","combatiente (0.72), suboficial (0.68), sargent..."
2,ejercito,"italiano (0.90), regimiento (0.90), unirse (0....","ejército (0.81), Ejército (0.66), soldados (0...."
3,batalla,"little (0.94), big (0.93), horn (0.93), muhlbe...","Batalla (0.83), batallas (0.77), escaramuza (0..."


In [56]:
def vectorizar_documento_promedio(tokens, modelo, dim=300):
    """
    Calcula el vector de un documento promediando los embeddings
    de las palabras presentes en el vocabulario del modelo.
    """
    # si pasamos modelo_propio accedemos a .wv
    # si es KeyedVectors (SBW) directamente al objeto
    wv = modelo.wv if hasattr(modelo, 'wv') else modelo

    vectores_palabras = [wv[w] for w in tokens if w in wv]

    if len(vectores_palabras) == 0:
        # Vector nulo si ninguna palabra está en el vocabulario
        return np.zeros(dim)

    return np.mean(vectores_palabras, axis=0)

# Generamos los vectores de documento con el modelo SBW (o con el propio)
df['vector_w2v_sbw'] = [vectorizar_documento_promedio(tokens, modelo_sbw) for tokens in corpus_tokens]
df['vector_w2v_propio'] = [vectorizar_documento_promedio(tokens, modelo_propio) for tokens in corpus_tokens]

print(f"Dimensiones del vector generado: {df['vector_w2v_sbw'].iloc[0].shape}")

Dimensiones del vector generado: (300,)


El vector de cada documento se construye calculando el promedio de los embeddings correspondientes a todas las palabras del texto que están presentes en el vocabulario del modelo. Si una palabra no existe en el modelo, se descarta para no incorporar ruido, y en caso de que un documento no comparta ningún término conocido, se le asigna un vector nulo.

## C. Modelo de oración: SBERT
Generen embeddings con un modelo de oración multilingüe. Arranquen con `distiluse-base-multilingual-cased-v1`, el de la Unidad 2; si prueban otro de los que aparecen en la sección de MTEB del apunte, justifiquen por qué.

Reporten:
1. La dimensión
2. El límite de tokens del modelo
3. **Cuántos de sus documentos se truncan** por superarlo (este último dato el modelo no lo avisa: hay que ir a buscarlo)


In [ ]:
nombre_modelo = 'distiluse-base-multilingual-cased-v1'
modelo_SBERT = SentenceTransformer(nombre_modelo)

# Consultar dimensión y límite de tokens del modelo SBERT
dimension = modelo_SBERT.get_sentence_embedding_dimension()
max_tokens = modelo_SBERT.max_seq_length

print(f"--- Ficha técnica del modelo: {nombre_modelo} ---")
print(f"Dimensión de los embeddings: {dimension}")
print(f"Límite máximo de tokens (max_seq_length): {max_tokens}")

# Detectar documentos truncados
documentos = df['sinopsis'].fillna("").tolist() # Entiendo que hay que usar la columna "sinopsis"

# El tokenizer parte las palabras en sub-tokens
tokenizer = modelo_SBERT.tokenizer

conteo_tokens = []
truncados = 0

for doc in documentos:
    # Tokenizamos sin truncar para conocer la longitud real en tokens del modelo
    tokens = tokenizer.encode(doc, truncation=False)
    longitud = len(tokens)
    conteo_tokens.append(longitud)

    if longitud > max_tokens:
        truncados += 1

porcentaje_truncados = (truncados / len(documentos)) * 100

print(f"\n--- Análisis de truncamiento ({len(documentos)} documentos) ---")
print(f"Documentos que superan los {max_tokens} tokens: {truncados} ({porcentaje_truncados:.2f}%)")
print(f"Longitud máxima encontrada: {max(conteo_tokens)} tokens")
print(f"Longitud promedio: {sum(conteo_tokens) / len(conteo_tokens):.1f} tokens")

# 4. Generar los embeddings reales de tu corpus
print("\nGenerando embeddings...")
embeddings = modelo_SBERT.encode(documentos, show_progress_bar=True, convert_to_numpy=True)
print(f"Matriz de embeddings generada con shape: {embeddings.shape}")

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1715.10it/s]
C:\Users\fethe\AppData\Local\Temp\ipykernel_28740\2951358348.py:5: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  dimension = modelo_SBERT.get_sentence_embedding_dimension()
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (253 > 128). Running this sequence through the model will result in indexing errors


--- Ficha técnica del modelo: distiluse-base-multilingual-cased-v1 ---
Dimensión de los embeddings: 512
Límite máximo de tokens (max_seq_length): 128

--- Análisis de truncamiento (150 documentos) ---
Documentos que superan los 128 tokens: 141 (94.00%)
Longitud máxima encontrada: 583 tokens
Longitud promedio: 252.1 tokens

Generando embeddings...


Batches: 100%|██████████| 5/5 [00:03<00:00,  1.48it/s]

Matriz de embeddings generada con shape: (150, 512)
